In [1]:
pip uninstall keras -y

Found existing installation: keras 3.8.0
Uninstalling keras-3.8.0:
  Successfully uninstalled keras-3.8.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --no-index --find-links /kaggle/input/hms-dependencies keras

Looking in links: /kaggle/input/hms-dependencies
ERROR: Could not find a version that satisfies the requirement keras (from versions: none)
ERROR: No matching distribution found for keras
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --no-index --find-links /kaggle/input/hms-dependencies sktime

Looking in links: /kaggle/input/hms-dependencies
ERROR: Could not find a version that satisfies the requirement sktime (from versions: none)
ERROR: No matching distribution found for sktime
Note: you may need to restart the kernel to use updated packages.


In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
from sklearn import preprocessing
import multiprocessing as mp
from pathos.multiprocessing import Pool

import tqdm
from scipy import signal

from sktime.classification.deep_learning.lstmfcn import LSTMFCNClassifier
from sktime.regression.deep_learning.mcdcnn import MCDCNNRegressor
from sktime.networks.mcdcnn import MCDCNNNetwork

from sktime.regression.base import BaseRegressor

import tensorflow as tf

import pickle
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

ModuleNotFoundError: No module named 'sktime'

In [5]:
import keras
keras.__version__

ModuleNotFoundError: No module named 'keras'

In [6]:
lstmfcn_cls = LSTMFCNClassifier()

lstmfcn_cls.model_ = tf.keras.models.load_model("/kaggle/input/hms_trained_lstmfcn/keras/v3/1/model.keras")
lstmfcn_cls.classes_ = [0, 1, 2, 3, 4, 5]

NameError: name 'LSTMFCNClassifier' is not defined

In [7]:
test_master = pd.read_csv("/kaggle/input/hms-harmful-brain-activity-classification/test.csv")


In [8]:
pairs = [['Fp1', 'F7'], ['F7', 'T3'], ['T3', 'T5'], 
         ['T5', 'O1'], ['Fp2', 'F8'], ['F8', 'T4'], 
         ['T4', 'T6'], ['T6', 'O2'], ['Fp1', 'F3'], 
        ['F3', 'C3'], ['C3', 'P3'], ['P3', 'O1'], 
        ['Fp2', 'F4'], ['F4', 'C4'], ['C4', 'P4'], 
         ['P4', 'O2'], ['Fz', 'Cz'], ['Cz', 'Pz']]

def get_features_for_entry(ir, q=2):
    
    r = ir[1]
    
    ex_eeg = pd.read_parquet("../input/hms-harmful-brain-activity-classification/test_eegs/{}.parquet".format(r['eeg_id']))
    ex_spec = pd.read_parquet("../input/hms-harmful-brain-activity-classification/test_spectrograms/{}.parquet".format(r['spectrogram_id']))
    
    eeg = ex_eeg
    eeg_subsample = eeg.iloc[20*200:30*200]
    times = np.arange(0, eeg_subsample.shape[0]/200, 1/200.) - 5.
    
    middle_50_spec = ex_spec.loc[(ex_spec.time>=275)
                     &(ex_spec.time<325)]
 
    
    output_eeg_X = np.array([])
    
    for p in pairs:
        row = np.nan_to_num(eeg_subsample[p[0]].values - eeg_subsample[p[1]].values)
        #downsample
        row = signal.decimate(row, q)
        
        if output_eeg_X.shape[0] == 0:
            output_eeg_X = preprocessing.normalize([row])[0]
        else:
            output_eeg_X = np.vstack((output_eeg_X, preprocessing.normalize([row])[0]))
    
            
    return pd.Series({"X_eeg": output_eeg_X, 
                      'spec_values': middle_50_spec.iloc[:, 1:].values})

In [9]:
for item in test_master.iterrows():
    res = get_features_for_entry(item)

In [10]:
print(res)

X_eeg          [[0.008920136574566154, 0.009011775038079222, ...
spec_values    [[641.43, 946.95, 1009.92, 948.96, 827.12, 308...
dtype: object


In [11]:
if len(res['X_eeg'].shape) == 2:
     prediction = lstmfcn_cls._predict_proba(np.array([res['X_eeg']]))
else:
     prediction = lstmfcn_cls._predict_proba(np.array(res['X_eeg'].values.tolist()))

NameError: name 'lstmfcn_cls' is not defined

In [12]:
col_names=["eeg_id", "seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]

normed = np.round(prediction, 4)

rows = []

for i in range(test_master.shape[0]):
     rows.append(np.append(test_master['eeg_id'].values[i], normed/normed.sum()))


result_df = pd.DataFrame(rows, columns = col_names)

display(result_df)
result_df.to_csv("submission.csv", index=False)

NameError: name 'prediction' is not defined

In [13]:
###Was going to try to reweight somehow...
#model i trained refuses to load, so may need to try something else